# 1. Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score, confusion_matrix, roc_curve, auc
)
from sklearn.preprocessing import StandardScaler

plt.style.use("ggplot")
sns.set_palette("viridis")

# 2. Carga de datos

In [ ]:
eur = pd.read_csv("europa_final.csv")
esp = pd.read_csv("espana_final.csv")

print(f"Europa: {eur.shape}")
print(f"España: {esp.shape}")
eur.head()

# 3. Preprocesamiento

In [ ]:
# Variables predictoras y objetivo
features = [
    "desempleo", "horas_hab_trabajadas", "poblacion_neet",
    "empleo", "parados_larga_dur", "region_codificada"
]
target = "absentismo"

# Dataset limpio sin nulos
df_eu = eur[features + [target, "pais", "año", "region"]].dropna()
print(f"Observaciones válidas: {df_eu.shape[0]}")

# Estandarización
scaler = StandardScaler()
X = scaler.fit_transform(df_eu[features])
y = df_eu[target].values

# División train/test 80-20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}")

## 3.1 Función de evaluación común

In [ ]:
def mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

nombres_feat = ["Desempleo","Horas hab.","Pob. NEET","Empleo","Paro larga dur.","Región"]

def evaluar(nombre, modelo, cv=5):
    modelo.fit(X_train, y_train)
    p_tr = modelo.predict(X_train)
    p_te = modelo.predict(X_test)
    cv_scores = cross_val_score(modelo, X, y, cv=cv, scoring="r2")
    sep = "=" * 50
    print(f"\n{sep}")
    print(f"Modelo: {nombre}")
    print(f"  R2    train={r2_score(y_train,p_tr):.4f}  test={r2_score(y_test,p_te):.4f}")
    print(f"  MAE   train={mean_absolute_error(y_train,p_tr):.4f}  test={mean_absolute_error(y_test,p_te):.4f}")
    print(f"  RMSE  train={np.sqrt(mean_squared_error(y_train,p_tr)):.4f}  test={np.sqrt(mean_squared_error(y_test,p_te)):.4f}")
    print(f"  MAPE  train={mape(y_train,p_tr):.2f}%  test={mape(y_test,p_te):.2f}%")
    print(f"  CV R2 = {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
    return modelo, p_te

# 4. Modelo no supervisado: K-Means clustering

## 4.1 Selección del número de clusters

In [ ]:
inertias, silhouettes = [], []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(list(K_range), inertias, marker="o", lw=2)
ax1.axvline(4, color="red", linestyle="--", label="k=4 seleccionado")
ax1.set_xlabel("Número de clusters (k)")
ax1.set_ylabel("Inercia")
ax1.set_title("Método del codo")
ax1.legend()
ax2.plot(list(K_range), silhouettes, marker="s", color="green", lw=2)
ax2.axvline(4, color="red", linestyle="--", label="k=4 seleccionado")
ax2.set_xlabel("Número de clusters (k)")
ax2.set_ylabel("Silhouette score")
ax2.set_title("Coeficiente de silueta")
ax2.legend()
plt.suptitle("Figura 1. Selección de k en K-Means")
plt.tight_layout()
plt.show()
for k, s in zip(K_range, silhouettes):
    print(f"k={k}: silhouette={s:.4f}")

## 4.2 Entrenamiento K-Means (k=4)

In [ ]:
km4 = KMeans(n_clusters=4, random_state=42, n_init=10)
df_eu = df_eu.copy()
df_eu["cluster"] = km4.fit_predict(X)

print("Medias por cluster:")
print(df_eu.groupby("cluster")[features + [target]].mean().round(2))

In [ ]:
palette = {0:"#2E86C1", 1:"#27AE60", 2:"#E67E22", 3:"#8E44AD"}

fig, ax = plt.subplots(figsize=(10, 7))
for cl in sorted(df_eu["cluster"].unique()):
    d = df_eu[df_eu["cluster"] == cl]
    ax.scatter(d["desempleo"], d["absentismo"],
               color=palette[cl], alpha=0.55, s=55,
               label=f"Cluster {cl}", edgecolors="white", lw=0.3)
esp_eu = df_eu[df_eu["pais"] == "España"]
ax.scatter(esp_eu["desempleo"], esp_eu["absentismo"],
           color="black", s=100, zorder=6, marker="*",
           label="España", edgecolors="red", lw=1.5)
ax.set_xlabel("Tasa de desempleo (%)")
ax.set_ylabel("Tasa de absentismo (%)")
ax.legend(fontsize=9)
ax.set_title("Figura 2. Segmentación K-Means (k=4): Europa 2009-2025")
plt.tight_layout()
plt.show()

In [ ]:
cl_abs = df_eu.groupby("cluster")["absentismo"].mean().sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([f"Cluster {i}" for i in cl_abs.index],
              cl_abs.values, color=[palette[i] for i in cl_abs.index], alpha=0.85)
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.1,
            f"{b.get_height():.1f}%", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Absentismo medio (%)")
ax.set_title("Figura 3. Absentismo medio por cluster K-Means")
plt.tight_layout()
plt.show()

# 5. Modelos de regresión supervisada

## 5.1 Regresión Lineal

In [ ]:
rl = LinearRegression()
rl, pred_rl = evaluar("Regresión Lineal", rl)

## 5.2 Ridge regression (λ=1.0)

In [ ]:
ri = Ridge(alpha=1.0)
ri, pred_ri = evaluar("Ridge (lambda=1.0)", ri)

# Comparativa de coeficientes
x_pos = np.arange(len(nombres_feat)); w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x_pos - w/2, rl.coef_, w, label="Regresión Lineal", alpha=0.85)
ax.bar(x_pos + w/2, ri.coef_, w, label="Ridge (lambda=1)", alpha=0.85, color="red")
ax.axhline(0, color="black", lw=1)
ax.set_xticks(x_pos)
ax.set_xticklabels(nombres_feat, fontsize=9)
ax.set_ylabel("Coeficiente")
ax.legend()
ax.set_title("Figura 11. Coeficientes Regresión Lineal vs Ridge")
plt.tight_layout()
plt.show()

## 5.3 Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200, max_depth=12, min_samples_leaf=2, random_state=42
)
rf, pred_rf = evaluar("Random Forest", rf)

In [ ]:
# Importancia de variables
fi = pd.Series(rf.feature_importances_, index=nombres_feat).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
colors_fi = ["#C0392B" if v == fi.max() else "#2E86C1" for v in fi.values]
ax.barh(fi.index, fi.values, color=colors_fi, alpha=0.85)
ax.set_xlabel("Importancia relativa")
ax.set_title("Figura 4. Importancia de variables — Random Forest")
plt.tight_layout()
plt.show()

In [ ]:
# Curvas de aprendizaje
rf_lc = RandomForestRegressor(
    n_estimators=200, max_depth=12, min_samples_leaf=2, random_state=42
)
train_sizes, train_scores, val_scores = learning_curve(
    rf_lc, X, y, cv=5, scoring="r2",
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1
)
ts_mean = train_scores.mean(axis=1); ts_std = train_scores.std(axis=1)
vs_mean = val_scores.mean(axis=1);   vs_std = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, ts_mean, "o-", lw=2.5, label="Train")
ax.fill_between(train_sizes, ts_mean-ts_std, ts_mean+ts_std, alpha=0.15)
ax.plot(train_sizes, vs_mean, "s-", color="green", lw=2.5, label="Validación cruzada")
ax.fill_between(train_sizes, vs_mean-vs_std, vs_mean+vs_std, alpha=0.15, color="green")
ax.axhline(0, color="gray", linestyle=":", lw=1)
ax.set_xlabel("Tamaño del conjunto de entrenamiento")
ax.set_ylabel("R2")
ax.legend()
ax.set_title("Figura 10. Curvas de aprendizaje — Random Forest")
plt.tight_layout()
plt.show()

## 5.4 Gradient Boosting Regressor

In [ ]:
gb = GradientBoostingRegressor(
    n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42
)
gb, pred_gb = evaluar("Gradient Boosting", gb)

In [ ]:
# Predicción media anual vs valor real europeo
df_eu["pred_gb"] = gb.predict(X)
evol = df_eu.groupby("año")[["absentismo","pred_gb"]].mean()

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(evol.index, evol["absentismo"], marker="o", lw=2.5, label="Absentismo real")
ax.plot(evol.index, evol["pred_gb"], marker="s", lw=2.5,
        linestyle="--", label="Predicción Gradient Boosting")
ax.fill_between(evol.index, evol["absentismo"], evol["pred_gb"], alpha=0.1)
ax.set_ylabel("Absentismo medio (%)")
ax.legend()
ax.set_xticks(evol.index)
ax.set_xticklabels(evol.index, rotation=45)
ax.set_title("Figura 12. Predicción media anual Gradient Boosting vs. valor real")
plt.tight_layout()
plt.show()

In [ ]:
# Análisis de residuos — Gradient Boosting
residuos = y_test - pred_gb
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.scatter(pred_gb, residuos, alpha=0.5, s=40, edgecolors="white", lw=0.3)
ax1.axhline(0, color="black", linestyle="--", lw=1.5)
ax1.set_xlabel("Valores predichos")
ax1.set_ylabel("Residuos")
ax1.set_title("Gráfico de residuos")
sns.histplot(residuos, kde=True, ax=ax2, alpha=0.6)
ax2.axvline(0, color="black", linestyle="--", lw=1.5)
ax2.set_xlabel("Residuos")
ax2.set_ylabel("Frecuencia")
ax2.set_title("Distribución de residuos")
fig.suptitle("Figura 6. Análisis de residuos — Gradient Boosting")
plt.tight_layout()
plt.show()

## 5.5 Red Neuronal: perceptrón multicapa (MLP)

In [ ]:
nn = MLPRegressor(
    hidden_layer_sizes=(64, 32, 16),
    max_iter=3000,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    learning_rate_init=0.001
)
nn, pred_nn = evaluar("Red Neuronal (MLP)", nn)

In [ ]:
# Valores reales vs predichos — MLP
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, pred_nn, alpha=0.5, s=40, edgecolors="white", lw=0.3)
mn = min(y_test.min(), pred_nn.min())
mx = max(y_test.max(), pred_nn.max())
ax.plot([mn,mx],[mn,mx],"--", color="black", lw=1.5, label="Predicción perfecta")
ax.set_xlabel("Valor real (%)")
ax.set_ylabel("Valor predicho (%)")
ax.legend()
ax.set_title("Figura 13. Red Neuronal MLP: reales vs predichos")
plt.tight_layout()
plt.show()

# 6. Comparativa de modelos

In [ ]:
resultados = pd.DataFrame({
    "Modelo": ["Reg. Lineal","Ridge (lam=1)","Random Forest","Gradient Boosting","MLP"],
    "R2_test":   [r2_score(y_test, p) for p in [pred_rl,pred_ri,pred_rf,pred_gb,pred_nn]],
    "MAE_test":  [mean_absolute_error(y_test, p) for p in [pred_rl,pred_ri,pred_rf,pred_gb,pred_nn]],
    "RMSE_test": [np.sqrt(mean_squared_error(y_test, p)) for p in [pred_rl,pred_ri,pred_rf,pred_gb,pred_nn]],
    "MAPE_test": [mape(y_test, p) for p in [pred_rl,pred_ri,pred_rf,pred_gb,pred_nn]],
})
print(resultados.round(4).to_string(index=False))

In [ ]:
# Gráfico comparativo
nombres = resultados["Modelo"].tolist()
r2s   = resultados["R2_test"].tolist()
maes  = resultados["MAE_test"].tolist()
rmses = resultados["RMSE_test"].tolist()
x = np.arange(len(nombres)); w = 0.25

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
bars = ax1.bar(x, r2s, w*2.5, alpha=0.85)
ax1.set_xticks(x)
ax1.set_xticklabels([n.replace(" ","\n") for n in nombres], fontsize=8)
ax1.set_ylabel("R2 (test)")
ax1.set_ylim(0, 1)
ax1.set_title("R2 en conjunto de test")
for b in bars:
    ax1.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
             f"{b.get_height():.3f}", ha="center", va="bottom", fontsize=8)
ax2.bar(x - w/2, maes,  w, label="MAE",  alpha=0.85)
ax2.bar(x + w/2, rmses, w, label="RMSE", alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels([n.replace(" ","\n") for n in nombres], fontsize=8)
ax2.set_ylabel("Error (pp)")
ax2.legend()
ax2.set_title("MAE y RMSE en conjunto de test")
fig.suptitle("Figura 7. Comparativa de métricas entre modelos")
plt.tight_layout()
plt.show()

In [ ]:
# Reales vs predichos — cuatro modelos
preds_list = [(pred_rl,"Reg. Lineal","#2E86C1"),
              (pred_rf,"Random Forest","#27AE60"),
              (pred_gb,"Gradient Boosting","#16A085"),
              (pred_nn,"MLP","#8E44AD")]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for (pred, name, col), ax in zip(preds_list, axes.flatten()):
    ax.scatter(y_test, pred, alpha=0.5, color=col, s=30,
               edgecolors="white", lw=0.3)
    mn = min(y_test.min(), pred.min())
    mx = max(y_test.max(), pred.max())
    ax.plot([mn,mx],[mn,mx],"--", color="black", lw=1.5)
    ax.set_xlabel("Valor real (%)")
    ax.set_ylabel("Valor predicho (%)")
    r2 = r2_score(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    ax.set_title(f"{name}  R2={r2:.3f}  MAE={mae:.2f}")
fig.suptitle("Figura 5. Valores reales vs. predichos — cuatro modelos")
plt.tight_layout()
plt.show()

## 6.1 Validación cruzada (k=5)

In [ ]:
modelos_cv = {
    "Reg.\nLineal":       LinearRegression(),
    "Ridge\n(lam=1)":    Ridge(alpha=1.0),
    "Random\nForest":    RandomForestRegressor(n_estimators=200, max_depth=12,
                              min_samples_leaf=2, random_state=42),
    "Gradient\nBoosting": GradientBoostingRegressor(n_estimators=200,
                              learning_rate=0.05, max_depth=4, random_state=42),
    "Red\nNeuronal":     MLPRegressor(hidden_layer_sizes=(64,32,16),
                              max_iter=2000, random_state=42, early_stopping=True),
}
cv_results = {nombre: cross_val_score(m, X, y, cv=5, scoring="r2")
              for nombre, m in modelos_cv.items()}

fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(cv_results.values(), labels=cv_results.keys(),
                patch_artist=True,
                medianprops=dict(color="white", linewidth=2))
ax.axhline(0, color="black", linestyle="--", lw=1, alpha=0.7)
ax.set_ylabel("R2 (validación cruzada, k=5)")
ax.set_title("Figura 14. Distribución de R2 en validación cruzada por modelo")
plt.tight_layout()
plt.show()

# 7. Clasificador binario de absentismo

In [ ]:
# Variable binaria: alto (1) si absentismo > mediana, bajo (0) si no
umbral = df_eu["absentismo"].median()
print(f"Mediana europea del absentismo: {umbral:.2f}%")

y_bin = (df_eu[target] > umbral).astype(int).values
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    X, y_bin, test_size=0.2, random_state=42
)
clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(Xc_tr, yc_tr)
y_prob   = clf.predict_proba(Xc_te)[:, 1]
y_pred_c = clf.predict(Xc_te)
cm       = confusion_matrix(yc_te, y_pred_c)
fpr, tpr, _ = roc_curve(yc_te, y_prob)
auc_score   = auc(fpr, tpr)
print(f"Accuracy: {(cm[0,0]+cm[1,1])/cm.sum():.4f}")
print(f"AUC:      {auc_score:.4f}")
print(f"Matriz de confusion:\n{cm}")

In [ ]:
# Curva ROC y matriz de confusión
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(fpr, tpr, lw=2.5, label=f"AUC = {auc_score:.3f}")
ax1.plot([0,1],[0,1],"--", color="gray", lw=1.5)
ax1.set_xlabel("Tasa de falsos positivos")
ax1.set_ylabel("Tasa de verdaderos positivos")
ax1.set_title("Curva ROC — Clasificador Random Forest")
ax1.legend()
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax2,
            xticklabels=["Bajo","Alto"], yticklabels=["Bajo","Alto"],
            cbar=False, linewidths=0.5)
ax2.set_xlabel("Prediccion")
ax2.set_ylabel("Real")
ax2.set_title("Matriz de confusion")
fig.suptitle("Figura 8. Clasificador binario: curva ROC y matriz de confusion")
plt.tight_layout()
plt.show()

# 8. Validación con España: la brecha estructural

In [ ]:
# Reentrenar Random Forest con variables comunes entre Europa y España
feats_comunes = ["desempleo", "horas_hab_trabajadas", "parados_larga_dur", "empleo"]

df_eu_val = eur[feats_comunes + ["absentismo"]].dropna()
sc2 = StandardScaler()
X_eu2 = sc2.fit_transform(df_eu_val[feats_comunes])
y_eu2 = df_eu_val["absentismo"].values

rf2 = RandomForestRegressor(
    n_estimators=200, max_depth=12, min_samples_leaf=2, random_state=42
)
rf2.fit(X_eu2, y_eu2)
print("Modelo reentrenado con features comunes.")

In [ ]:
# Aplicar el modelo europeo sobre España
df_esp_val = esp[feats_comunes + ["absentismo_total", "año"]].dropna()

X_esp    = sc2.transform(df_esp_val[feats_comunes])
pred_esp = rf2.predict(X_esp)
real_esp = df_esp_val["absentismo_total"].values
años_esp = df_esp_val["año"].values

gap = pred_esp - real_esp
print(f"Gap medio (modelo - real): {gap.mean():.2f} pp")
print("\nAño | Real (%) | Pred (%) | Gap (pp)")
for a, r, p, g in zip(años_esp, real_esp, pred_esp, gap):
    print(f"{int(a)}  {r:.2f}      {p:.2f}      {g:+.2f}")

In [ ]:
# Visualización: valores absolutos + tendencias normalizadas
def znorm(x): return (x - x.mean()) / x.std()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

ax1.plot(años_esp, real_esp, marker="o", color="red", lw=2.5,
         label="Absentismo real (INE)")
ax1.plot(años_esp, pred_esp, marker="s", color="blue", lw=2.5,
         linestyle="--", label="Predicción modelo europeo")
ax1.fill_between(años_esp, pred_esp, real_esp,
                 where=(pred_esp > real_esp), alpha=0.15, color="blue", label="Gap")
ax1.set_ylabel("% Absentismo")
ax1.legend()
ax1.set_ylim(0, 15)
ax1.set_xticks(años_esp)
ax1.set_xticklabels(años_esp, rotation=45)
ax1.set_title("Absentismo real vs. predicción del modelo europeo")

ax2.plot(años_esp, znorm(real_esp), marker="o", color="red", lw=2.5,
         label="Real (normalizado)")
ax2.plot(años_esp, znorm(pred_esp), marker="s", color="blue", lw=2.5,
         linestyle="--", label="Predicción (normalizada)")
ax2.axhline(0, color="gray", lw=1, linestyle=":")
ax2.set_ylabel("Z-score")
ax2.legend()
ax2.set_xticks(años_esp)
ax2.set_xticklabels(años_esp, rotation=45)
ax2.set_title("Tendencias normalizadas (Z-score)")

fig.suptitle("Figura 9. Validación del modelo europeo sobre España: brecha estructural")
plt.tight_layout()
plt.show()

# 9. Exportar datasets finales

In [ ]:
eur.to_csv("europa_final.csv", index=False)
esp.to_csv("espana_final.csv", index=False)
print("Datasets exportados correctamente.")